# Claims Investigation Agent — Eval-Driven Development

**Workshop flow**
1. Setup — install dependencies, set API keys, clone repo
2. Build & run the agent — see the trace in LangSmith
3. Spot the problem — agent hallucinates in its rationale
4. Apply the grounding evaluator — catch the failure systematically
5. Fix the prompt — one line change
6. Re-run & compare — grounding score improves

## 1 · Setup

In [ ]:
import sys, os

# Add repo root to sys.path so 'data' and 'evals' are importable
repo_root = os.path.abspath(os.path.join(os.path.dirname("__file__"), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

assert os.path.exists(os.path.join(repo_root, "data/sources/policy_docs.md")), \
    f"Expected to find data/sources/policy_docs.md under {repo_root}"
print("Repo root:", repo_root)

# Run once if packages are missing. Skip if you installed via: uv sync
# !pip install -q langchain-openai langgraph langsmith openevals python-dotenv

In [ ]:
# Reads OPENAI_API_KEY and LANGSMITH_API_KEY from .env at the repo root.
from dotenv import load_dotenv
load_dotenv()

import os
os.environ.setdefault('LANGSMITH_TRACING', 'true')
os.environ.setdefault('LANGSMITH_PROJECT', 'claims-workshop')
print('OPENAI_API_KEY set:   ', bool(os.getenv('OPENAI_API_KEY')))
print('LANGSMITH_API_KEY set:', bool(os.getenv('LANGSMITH_API_KEY')))

## 2 · Build the agent

Four tools — one per data source. The agent decides which ones to call based on the claim.

In [ ]:
from langchain_core.tools import tool
from data.loaders import load_policy_docs, load_claims_history, load_weather_data, load_repair_estimate

@tool
def search_policy_docs() -> str:
    """Retrieve insurance policy clauses, coverage conditions, thresholds, and exclusions.
    Always call this first."""
    return load_policy_docs()

@tool
def query_claims_history(claimant_id: str) -> list:
    """Retrieve prior claims history for a claimant ID.
    Always call this to check for repeat claims."""
    return load_claims_history(claimant_id)

@tool
def query_weather_data(incident_date: str, location: str) -> dict:
    """Retrieve historical weather for an incident date (YYYY-MM-DD) and city.
    Call when the cause could be weather-related. Clause 2 thresholds: 40mm or 90 km/h."""
    return load_weather_data(incident_date, location)

@tool
def retrieve_repair_estimate(claim_id: str) -> dict:
    """Retrieve contractor repair estimate. Required by Clause 6 for claims over €10,000."""
    return load_repair_estimate(claim_id)

tools = [search_policy_docs, query_claims_history, query_weather_data, retrieve_repair_estimate]
print("Tools:", [t.name for t in tools])

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

# ⚠️ BEFORE: no instruction to cite retrieved content → hallucination risk
PROMPT_BEFORE = """You are a claims investigation assistant for EuroShield Insurance Group.

Investigation steps:
1. Always call search_policy_docs and query_claims_history.
2. Call query_weather_data if the cause could be weather-related.
3. Call retrieve_repair_estimate if the amount exceeds €10,000.

Then provide: coverage decision (covered/partial/excluded),
settlement recommendation (auto_settle/assign_adjuster/flag_for_investigation),
confidence (high/medium/low), and rationale."""

# ✅ AFTER: must quote tool outputs → grounded rationale
PROMPT_AFTER = """You are a claims investigation assistant for EuroShield Insurance Group.

Investigation steps:
1. Always call search_policy_docs and query_claims_history.
2. Call query_weather_data if the cause could be weather-related.
3. Call retrieve_repair_estimate if the amount exceeds €10,000.

Then provide: coverage decision (covered/partial/excluded),
settlement recommendation (auto_settle/assign_adjuster/flag_for_investigation),
confidence (high/medium/low), and rationale.
Your rationale must QUOTE the exact text returned by the tools —
do not assert facts not present in the tool outputs."""

llm   = ChatOpenAI(model="gpt-4o", temperature=0)
agent = create_agent(llm, tools)
print("Agent ready")

## 3 · Run the agent

Two claims — one clean baseline, one with fraud signals and ambiguous evidence.

In [ ]:
def run(claim: dict, prompt: str) -> str:
    """Run the agent and return its final response."""
    claim_text = "\n".join(f"{k}: {v}" for k, v in claim.items())
    state = agent.invoke({
        "messages": [
            SystemMessage(content=prompt),
            HumanMessage(content=claim_text),
        ]
    })
    return state["messages"][-1].content

In [ ]:
# Claim 1 — clean baseline (CLM003, Utrecht, €9,800, no prior history)
from evals.dataset import CLAIM_INPUTS
clean_claim = CLAIM_INPUTS[3]  # CLAIM-2022-0441

print("Input:", clean_claim["claim_id"], "|", clean_claim["reported_cause"])
print()
response_clean = run(clean_claim, PROMPT_BEFORE)
print(response_clean)

In [ ]:
# Claim 2 — fraud signals (CLM007, Amsterdam, €28,900, weather data contradicts reported cause)
fraud_claim = CLAIM_INPUTS[0]  # CLAIM-2024-0891

print("Input:", fraud_claim["claim_id"], "|", fraud_claim["reported_cause"])
print()
response_fraud = run(fraud_claim, PROMPT_BEFORE)
print(response_fraud)

## 4 · Apply the grounding evaluator

Is the rationale supported by what the tools actually returned?

We ask an LLM judge to compare the agent's response against the tool outputs in the trace.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

GROUNDING_PROMPT = """You are evaluating whether an AI agent's response is grounded
in the evidence it actually retrieved from its tools.

## Tool outputs retrieved during investigation:
{retrieved_content}

## Agent's final response:
{response}

Is the response grounded in the tool outputs above?
- GROUNDED: every factual claim traces to specific retrieved content.
- PARTIALLY_GROUNDED: mostly supported, but one claim is vague or not traceable.
- HALLUCINATED: the response asserts facts not present in the retrieved content.

Reply with one of: GROUNDED, PARTIALLY_GROUNDED, HALLUCINATED
Then one sentence explaining which claim is unsupported (if any)."""

_judge_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def call_judge(retrieved_content: str, response: str) -> dict:
    prompt = GROUNDING_PROMPT.format(retrieved_content=retrieved_content, response=response)
    result = _judge_llm.invoke([HumanMessage(content=prompt)])
    text = result.content.strip()
    first_line = text.split("\n")[0].upper()
    comment = text.split("\n", 1)[1].strip() if "\n" in text else ""
    return {"score": first_line, "comment": comment}

print("Judge ready")

In [ ]:
def get_tool_outputs(claim: dict, prompt: str):
    """Run the agent and return (retrieved_content, response)."""
    claim_text = "\n".join(f"{k}: {v}" for k, v in claim.items())
    state = agent.invoke({
        "messages": [
            SystemMessage(content=prompt),
            HumanMessage(content=claim_text),
        ]
    })
    tool_outputs = {}
    for msg in state["messages"]:
        if getattr(msg, "type", None) == "tool":
            tool_outputs[msg.name] = str(msg.content)
    retrieved = "\n\n".join(f"### {k}\n{v}" for k, v in tool_outputs.items())
    response  = state["messages"][-1].content
    return retrieved, response


def score_grounding(claim: dict, prompt: str) -> None:
    retrieved, response = get_tool_outputs(claim, prompt)
    result  = call_judge(retrieved, response)
    verdict = result["score"]
    score   = 1.0 if ("GROUNDED" in verdict and "PARTIAL" not in verdict and "HALL" not in verdict) \
              else 0.5 if "PARTIAL" in verdict else 0.0
    label   = {1.0: "✅ GROUNDED", 0.5: "⚠️  PARTIAL", 0.0: "❌ HALLUCINATED"}[score]
    print(f"Score: {label}")
    print(f"Comment: {result['comment']}")
    print()
    print("--- Response ---")
    print(response)

In [ ]:
# Evaluate the fraud claim with PROMPT_BEFORE
print("=== BEFORE (no grounding instruction) ===")
score_grounding(fraud_claim, PROMPT_BEFORE)

## 5 · Fix the prompt & re-evaluate

One line added to the prompt: *"Your rationale must QUOTE the exact text returned by the tools."*

In [ ]:
print("=== AFTER (grounding instruction added) ===")
score_grounding(fraud_claim, PROMPT_AFTER)

## 6 · Run across all claims with LangSmith evaluate()

Scale the same evaluator across the full dataset and compare experiments in LangSmith.

In [ ]:
from langsmith import Client
from langsmith.evaluation import evaluate

DATASET_NAME = "claims-investigation-workshop"
client = Client()

# Push dataset once
if DATASET_NAME not in {d.name for d in client.list_datasets()}:
    ds = client.create_dataset(dataset_name=DATASET_NAME)
    for claim in CLAIM_INPUTS:
        client.create_example(inputs={"claim": claim}, dataset_id=ds.id)
    print(f"Created dataset with {len(CLAIM_INPUTS)} examples.")
else:
    print(f"Dataset '{DATASET_NAME}' already exists.")

In [ ]:
# Change prompt_version and active_prompt to compare before vs after
active_prompt   = PROMPT_AFTER   # ← swap to PROMPT_BEFORE for the failure run
prompt_version  = "after"        # ← swap to "before"

def target(inputs: dict) -> dict:
    claim_text = "\n".join(f"{k}: {v}" for k, v in inputs["claim"].items())
    state = agent.invoke({
        "messages": [
            SystemMessage(content=active_prompt),
            HumanMessage(content=claim_text),
        ]
    })
    return {"messages": state["messages"]}

def grounding_evaluator(run, example):
    tool_outputs = {}
    for msg in run.outputs.get("messages", []):
        if getattr(msg, "type", None) == "tool":
            tool_outputs[msg.name] = str(msg.content)
    retrieved = "\n\n".join(f"### {k}\n{v}" for k, v in tool_outputs.items()) or "No tool outputs."
    response  = run.outputs["messages"][-1].content

    result  = call_judge(retrieved, response)
    verdict = result["score"]
    score   = 1.0 if ("GROUNDED" in verdict and "PARTIAL" not in verdict and "HALL" not in verdict) \
              else 0.5 if "PARTIAL" in verdict else 0.0
    return {"key": "grounding", "score": score, "comment": result["comment"]}

results = evaluate(
    target,
    data=DATASET_NAME,
    evaluators=[grounding_evaluator],
    experiment_prefix="grounding",
    metadata={"prompt_version": prompt_version},
)

scores = [r["evaluation_results"]["results"][0].score for r in results]
labels = {1.0: "✅ GROUNDED", 0.5: "⚠️  PARTIAL", 0.0: "❌ HALLUCINATED"}
for claim, score in zip(CLAIM_INPUTS, scores):
    print(f"  {claim['claim_id']}  →  {labels.get(score, score)}")
print(f"\nMean grounding score: {sum(scores)/len(scores):.2f}")